# 05 · 超參數調校

XGBoost 有一大把旋鈕，手動試到死也試不完。這堂課用 **隨機搜尋（RandomizedSearchCV）** 系統性地找好設定，並認識更聰明的 **Optuna** 是什麼。

## 學習目標

- 知道 boosting 該優先調哪些參數
- 用 `RandomizedSearchCV` 在參數空間裡有效率地搜尋
- 知道 grid / random / 貝氏（Optuna）搜尋的差別

## 1. 該調哪些參數？（優先序）

1. `learning_rate` + `n_estimators`（一對，配 early stopping）
2. `max_depth`、`min_child_weight`（樹的複雜度）
3. `subsample`、`colsample_bytree`（隨機性）
4. `reg_lambda`、`reg_alpha`（正則化）

## 2. 隨機搜尋

參數組合是天文數字，grid search 全試不切實際。**隨機搜尋**在範圍內隨機抽 N 組來試，用同樣預算通常找到更好的解。

In [1]:
import numpy as np
from scipy.stats import randint, uniform
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

X, y = load_breast_cancer(return_X_y=True)

param_dist = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(2, 7),
    "learning_rate": uniform(0.01, 0.3),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
}
search = RandomizedSearchCV(
    XGBClassifier(eval_metric="logloss", random_state=42),
    param_dist, n_iter=25, cv=5, scoring="roc_auc", random_state=42, n_jobs=-1,
)
search.fit(X, y)
print("最佳 AUC:", round(search.best_score_, 4))
print("最佳參數:")
for k, v in search.best_params_.items():
    print(f"  {k}: {round(v, 3) if isinstance(v, float) else v}")

最佳 AUC: 0.9948
最佳參數:
  colsample_bytree: 0.98
  learning_rate: 0.3
  max_depth: 3
  n_estimators: 364
  subsample: 0.606


## 3. 更聰明的搜尋：Optuna

random search 是亂槍打鳥；**貝氏最佳化**（如 `optuna` 套件）會根據已試過的結果，**聰明地**決定下一組要試什麼，用更少次數找到更好的解。Kaggle 高手幾乎都用 Optuna。本課不展開（需另裝套件），但你該知道它存在——在 Colab 裡 `pip install optuna` 即可玩。

## 小結

- 調參優先序：學習率/樹數 → 樹複雜度 → 隨機性 → 正則化。
- `RandomizedSearchCV` 比 grid search 更划算。
- 進階用 **Optuna**（貝氏最佳化），用更少嘗試找到更好的參數。

## 練習

1. 把 `n_iter` 從 25 加到 60，最佳 AUC 有提升嗎？代價是什麼？
2. 在 Colab 裝 `optuna`，用它調同一個 XGBoost，比較找到的 AUC。

下一課，認識另外兩個 boosting 庫：**LightGBM 與 CatBoost**。